## Schema audit — findings from manual inspection

### Billing files (2022–2026)
- All files: 8 columns — `year, month, division, section, collection, demand, noofcans, category`
- Already in long format — one row per section-category per month
- No wide format reshaping needed (plan assumption was wrong — this is simpler)
- Row count growth: ~2,275 rows/month in 2022 → ~4,433 rows/month in 2026
- Growth cause: new sections added + old sections renamed (e.g. BAHADURPURA → BAHADURPURA (OLD))
- 2026 files are ~2x larger purely due to more sections, not schema change

### Water connection files (2023–2026)  
- 7 columns — `year, month, division, section, applied, approved, category`
- Category values are descriptive text (e.g. `INDIVIDUAL/DOMESTIC`), NOT the short codes used in billing
- Join strategy: join on `(year, month, division, section)` only — drop category from join key

### ETL implications
- No schema detection logic needed — single schema across all years
- Main cleaning challenges: section name drift, zero-demand rows, overcollection rows
- Water connection join will be approximate — section-level only

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project root is the Jupyter working directory
ROOT = Path().resolve()
RAW  = ROOT / "data" / "raw"

# Load all billing files
billing_files    = sorted(RAW.glob("billing_and_collection_report_*.csv"))
connection_files = sorted(RAW.glob("water_connection_report_*.csv"))

print(f"Billing files:    {len(billing_files)}")
print(f"Connection files: {len(connection_files)}")

# Concatenate all billing data
df = pd.concat([pd.read_csv(f) for f in billing_files], ignore_index=True)

print(f"\nShape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nFirst 5 rows:\n{df.head()}")
print(f"\nNull counts:\n{df.isnull().sum()}")

Billing files:    52
Connection files: 40

Shape: (127511, 8)

Dtypes:
year            int64
month           int64
division      float64
section        object
collection    float64
demand        float64
noofcans        int64
category       object
dtype: object

First 5 rows:
   year  month  division      section  collection      demand  noofcans  \
0  2022      1       1.0  BAHADURPURA  1179062.89   971293.50        68   
1  2022      1       1.0  BAHADURPURA    52204.50  7670296.00      4528   
2  2022      1       1.0  BAHADURPURA        0.00     3621.80         2   
3  2022      1       1.0  BAHADURPURA        0.00  1059235.45      1303   
4  2022      1       1.0  BAHADURPURA        0.00        0.00         4   

  category  
0        C  
1        D  
2       DM  
3       DS  
4       FS  

Null counts:
year            0
month           0
division      221
section        29
collection      0
demand          0
noofcans        0
category        0
dtype: int64


In [4]:
# ── Data quality assessment ──────────────────────────────────────────────────

# 1. Division nulls — what do these rows look like?
print("=== Rows with null division ===")
print(df[df['division'].isna()][['year','month','section','category','demand','collection']].head(10))

# 2. Section nulls
print("\n=== Rows with null section ===")
print(df[df['section'].isna()][['year','month','division','category','demand','collection']].head(10))

# 3. Zero demand rows
zero_demand = df[df['demand'] == 0]
print(f"\n=== Zero demand rows: {len(zero_demand)} ({len(zero_demand)/len(df)*100:.1f}%) ===")
print(zero_demand['category'].value_counts().head(10))

# 4. Overcollection rows (collection > demand)
overcollection = df[(df['demand'] > 0) & (df['collection'] > df['demand'])]
print(f"\n=== Overcollection rows: {len(overcollection)} ({len(overcollection)/len(df)*100:.1f}%) ===")
print(f"Max overcollection ratio: {(overcollection['collection']/overcollection['demand']).max():.2f}x")

# 5. Category distribution
print("\n=== Category value counts ===")
print(df['category'].value_counts())

=== Rows with null division ===
      year  month      section category  demand  collection
2267  2022      1  CHANDANAGAR        D     0.0         0.0
2268  2022      1  CHANDANAGAR       DM  1810.9         0.0
2269  2022      1   SAHEBNAGAR        C     0.0         0.0
2270  2022      1   SAHEBNAGAR       CH     0.0         0.0
2271  2022      1   SAHEBNAGAR        D     0.0         0.0
2272  2022      1   SAHEBNAGAR        N     0.0         0.0
2273  2022      1          NaN        D     0.0         0.0
4541  2022     10  CHANDANAGAR        D  5691.4         0.0
4542  2022     10  CHANDANAGAR       DM   776.1         0.0
4543  2022     10   SAHEBNAGAR        C     0.0         0.0

=== Rows with null section ===
       year  month  division category  demand  collection
2273   2022      1       NaN        D     0.0         0.0
4547   2022     10       NaN        D     0.0         0.0
6821   2022     11       NaN        D     0.0         0.0
9095   2022     12       NaN        D     0.

In [5]:
# ── Target variable: collection efficiency ───────────────────────────────────

# Step 1: Clean the dataframe
df_clean = df.copy()

# Drop rows where section is null (empty placeholder rows)
df_clean = df_clean[df_clean['section'].notna()].copy()

# Fill missing division with 0 (sentinel — section is still usable)
df_clean['division'] = df_clean['division'].fillna(0).astype(int)

# Step 2: Separate zero-demand rows (cannot compute efficiency)
zero_demand_df = df_clean[df_clean['demand'] == 0].copy()
df_valid = df_clean[df_clean['demand'] > 0].copy()

print(f"Total rows:       {len(df_clean):>7}")
print(f"Zero demand rows: {len(zero_demand_df):>7} ({len(zero_demand_df)/len(df_clean)*100:.1f}%)")
print(f"Valid rows:       {len(df_valid):>7} ({len(df_valid)/len(df_clean)*100:.1f}%)")

# Step 3: Compute efficiency and clip to [0, 2]
df_valid['efficiency'] = (df_valid['collection'] / df_valid['demand']).clip(0, 2)

# Step 4: Distribution summary
print(f"\n=== Efficiency distribution ===")
print(df_valid['efficiency'].describe())

print(f"\nPct with efficiency < 0.5:  {(df_valid['efficiency'] < 0.5).mean()*100:.1f}%")
print(f"Pct with efficiency 0.5–0.8: {((df_valid['efficiency'] >= 0.5) & (df_valid['efficiency'] < 0.8)).mean()*100:.1f}%")
print(f"Pct with efficiency >= 0.8:  {(df_valid['efficiency'] >= 0.8).mean()*100:.1f}%")

# Step 5: Rupee shortfall
df_valid['shortfall'] = (df_valid['demand'] - df_valid['collection']).clip(lower=0)
total_shortfall = df_valid['shortfall'].sum()
print(f"\nTotal rupee shortfall across all rows: ₹{total_shortfall:,.0f}")
print(f"Average monthly shortfall per row:     ₹{df_valid['shortfall'].mean():,.0f}")

Total rows:        127482
Zero demand rows:   27340 (21.4%)
Valid rows:         99688 (78.2%)

=== Efficiency distribution ===
count    99688.000000
mean         0.557414
std          0.498316
min          0.000000
25%          0.083021
50%          0.486814
75%          0.919920
max          2.000000
Name: efficiency, dtype: float64

Pct with efficiency < 0.5:  51.0%
Pct with efficiency 0.5–0.8: 17.7%
Pct with efficiency >= 0.8:  31.4%

Total rupee shortfall across all rows: ₹31,966,264,890
Average monthly shortfall per row:     ₹320,663


In [6]:
# ── Category group analysis ───────────────────────────────────────────────────

# Define super-category mapping
category_map = {
    'D': 'Domestic', 'DM': 'Domestic', 'M0': 'Domestic', 'M1': 'Domestic',
    'M2': 'Domestic', 'M3': 'Domestic', 'M4': 'Domestic', 'MS': 'Domestic',
    'DP': 'Domestic', 'T1': 'Domestic', 'T2': 'Domestic', 'T3': 'Domestic',
    'T4': 'Domestic',
    'DS': 'Slum', 'PS': 'Slum', 'GP': 'Slum',
    'C': 'Commercial', 'BC': 'Commercial', 'C1': 'Commercial',
    'FS': 'Commercial', 'XO': 'Commercial', 'X': 'Commercial',
    'I': 'Industrial', 'I1': 'Industrial', 'IW': 'Industrial',
    'G': 'GovtInstitutional', 'H': 'GovtInstitutional', 'RC': 'GovtInstitutional',
    'CH': 'GovtInstitutional', 'CB': 'GovtInstitutional', 'MB': 'GovtInstitutional',
    'N': 'GovtInstitutional', 'O': 'GovtInstitutional', 'S': 'GovtInstitutional',
    'V': 'GovtInstitutional'
}

df_valid['category_group'] = df_valid['category'].map(category_map).fillna('Other')

# Efficiency by category group
group_stats = df_valid.groupby('category_group').agg(
    median_efficiency=('efficiency', 'median'),
    mean_efficiency=('efficiency', 'mean'),
    total_demand=('demand', 'sum'),
    total_shortfall=('shortfall', 'sum'),
    row_count=('efficiency', 'count')
).sort_values('median_efficiency')

group_stats['pct_of_total_demand']    = group_stats['total_demand'] / group_stats['total_demand'].sum() * 100
group_stats['pct_of_total_shortfall'] = group_stats['total_shortfall'] / group_stats['total_shortfall'].sum() * 100

print("=== Efficiency and shortfall by category group ===\n")
print(group_stats[['median_efficiency','mean_efficiency',
                    'pct_of_total_demand','pct_of_total_shortfall',
                    'row_count']].round(3).to_string())

print(f"\n=== Rupee shortfall by category group ===")
for grp, row in group_stats.sort_values('total_shortfall', ascending=False).iterrows():
    print(f"{grp:<20} ₹{row['total_shortfall']:>18,.0f}  ({row['pct_of_total_shortfall']:.1f}% of total shortfall)")

=== Efficiency and shortfall by category group ===

                   median_efficiency  mean_efficiency  pct_of_total_demand  pct_of_total_shortfall  row_count
category_group                                                                                               
Slum                           0.014            0.074                2.964                   6.823       9900
GovtInstitutional              0.366            0.489                9.525                  15.867      13915
Domestic                       0.465            0.554               44.809                  65.788      55775
Commercial                     0.843            0.847               14.009                   7.211      11721
Industrial                     1.000            0.858               28.693                   4.312       8377

=== Rupee shortfall by category group ===
Domestic             ₹    21,029,876,122  (65.8% of total shortfall)
GovtInstitutional    ₹     5,072,194,985  (15.9% of total shortfal

## Core business finding

Total uncollected revenue (2022–2026): ₹31.97 billion

### The counterintuitive result
- Slum connections have the worst efficiency (1.4% median) but only 6.8% of shortfall
- Domestic connections (46.5% median efficiency) drive 65.8% = ₹21 billion of shortfall
- Reason: domestic demand volume (44.8% of total billed) overwhelms the efficiency gap
- Industrial connections pay reliably (100% median) — enforcement and metering work

### Strategic implication
A 10 percentage point improvement in domestic collection efficiency would recover
approximately ₹2.1 billion annually — more than the entire slum shortfall combined.
The dashboard should prioritize domestic sections with chronic low efficiency,
not just the lowest-efficiency sections by ratio.

In [8]:
# ── Seasonal pattern analysis ─────────────────────────────────────────────────

monthly = df_valid.groupby('month').agg(
    mean_efficiency=('efficiency', 'mean'),
    median_efficiency=('efficiency', 'median'),
    total_shortfall=('shortfall', 'sum')
).reset_index()

month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
monthly['month_name'] = monthly['month'].map(month_names)

print("=== Monthly efficiency pattern ===\n")
print(monthly[['month_name','mean_efficiency','median_efficiency','total_shortfall']].to_string(index=False))

print(f"\nMonsoon months (Jun-Sep) mean efficiency:     {monthly[monthly['month'].isin([6,7,8,9])]['mean_efficiency'].mean():.3f}")
print(f"Non-monsoon months mean efficiency:            {monthly[~monthly['month'].isin([6,7,8,9])]['mean_efficiency'].mean():.3f}")
print(f"March (financial year end) mean efficiency:   {monthly[monthly['month']==3]['mean_efficiency'].values[0]:.3f}")

=== Monthly efficiency pattern ===

month_name  mean_efficiency  median_efficiency  total_shortfall
       Jan         0.500497           0.439275     9.510161e+09
       Feb         0.508666           0.436889     2.863478e+09
       Mar         0.546208           0.466073     2.345167e+09
       Apr         0.522948           0.462899     2.500220e+09
       May         0.533252           0.451970     2.077028e+09
       Jun         0.551958           0.479142     2.002395e+09
       Jul         0.575028           0.500210     1.726991e+09
       Aug         0.549177           0.468888     1.874821e+09
       Sep         0.593197           0.527069     1.824941e+09
       Oct         0.638319           0.566087     1.913029e+09
       Nov         0.652125           0.578825     1.605476e+09
       Dec         0.571251           0.514699     1.722559e+09

Monsoon months (Jun-Sep) mean efficiency:     0.567
Non-monsoon months mean efficiency:            0.559
March (financial year end)

In [9]:
# ── Section-level chronic underperformers ─────────────────────────────────────

section_stats = df_valid.groupby(['section', 'category_group']).agg(
    mean_efficiency=('efficiency', 'mean'),
    std_efficiency=('efficiency', 'std'),
    total_demand=('demand', 'sum'),
    total_shortfall=('shortfall', 'sum'),
    months_observed=('efficiency', 'count'),
    pct_months_below_70=('efficiency', lambda x: (x < 0.7).mean())
).reset_index()

# Only keep section-category pairs with at least 12 months of data
section_stats = section_stats[section_stats['months_observed'] >= 12].copy()

print(f"Section-category pairs with 12+ months: {len(section_stats)}")

# Top 10 worst by total shortfall (domestic only — highest business value)
domestic_worst = (section_stats[section_stats['category_group'] == 'Domestic']
                  .sort_values('total_shortfall', ascending=False)
                  .head(10))

print("\n=== Top 10 domestic sections by total shortfall ===")
print(domestic_worst[['section','mean_efficiency','total_shortfall',
                       'pct_months_below_70','months_observed']].to_string(index=False))

# Chronic underperformers — below 50% efficiency more than 80% of months
chronic = section_stats[
    (section_stats['pct_months_below_70'] > 0.8) &
    (section_stats['total_demand'] > 1_000_000)
].sort_values('total_shortfall', ascending=False)

print(f"\n=== Chronic underperformers (>80% months below 70% efficiency, demand >₹1M) ===")
print(f"Count: {len(chronic)}")
print(chronic[['section','category_group','mean_efficiency',
               'total_shortfall','pct_months_below_70']].head(15).to_string(index=False))

Section-category pairs with 12+ months: 1203

=== Top 10 domestic sections by total shortfall ===
               section  mean_efficiency  total_shortfall  pct_months_below_70  months_observed
                  KPHB         0.585113     5.124223e+09             0.667808              292
           BHAGYANAGAR         0.685979     2.060142e+08             0.489726              292
     KONDAPUR (DIV 15)         0.768863     2.022719e+08             0.462633              281
         BANJARA HILLS         0.595489     1.873167e+08             0.637427              342
              MADHAPUR         0.736730     1.775239e+08             0.403587              223
        YELLAREDDYGUDA         0.474033     1.558486e+08             0.800000              260
VAISHALINAGAR (DIV 10)         0.412820     1.558442e+08             0.821114              341
    HAFEEZPET (DIV 15)         0.712917     1.544623e+08             0.449495              198
   SAHEBNAGAR (DIV 10)         0.542052     1.4

In [10]:
# ── KMeans risk tier clustering ───────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Use only 2022-2024 data for clustering (avoid leakage)
df_train_period = df_valid[df_valid['year'] <= 2024].copy()
df_train_period['efficiency'] = (df_train_period['collection'] / 
                                  df_train_period['demand']).clip(0, 2)
df_train_period['shortfall'] = (df_train_period['demand'] - 
                                 df_train_period['collection']).clip(lower=0)

# Build section-category profile using training period only
profile = df_train_period.groupby(['section', 'category_group']).agg(
    mean_efficiency=('efficiency', 'mean'),
    std_efficiency=('efficiency', 'std'),
    log_total_demand=('demand', lambda x: np.log1p(x.sum())),
    pct_months_below_70=('efficiency', lambda x: (x < 0.7).mean()),
    months_count=('efficiency', 'count')
).reset_index()

# Only cluster pairs with enough history
profile = profile[profile['months_count'] >= 6].copy()
profile['std_efficiency'] = profile['std_efficiency'].fillna(0)

print(f"Section-category pairs for clustering: {len(profile)}")

# Scale features
features = ['mean_efficiency', 'std_efficiency', 
            'log_total_demand', 'pct_months_below_70']
scaler = StandardScaler()
X = scaler.fit_transform(profile[features])

# Elbow method
inertias = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

print("\n=== Elbow method ===")
print(f"{'k':>3} {'inertia':>12} {'silhouette':>12}")
for k, inertia, sil in zip(K_range, inertias, silhouettes):
    print(f"{k:>3} {inertia:>12.1f} {sil:>12.4f}")

# Fit final model with k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
profile['cluster'] = kmeans.fit_predict(X)

# Interpret clusters
print("\n=== Cluster profiles ===")
cluster_summary = profile.groupby('cluster').agg(
    mean_efficiency=('mean_efficiency', 'mean'),
    std_efficiency=('std_efficiency', 'mean'),
    pct_months_below_70=('pct_months_below_70', 'mean'),
    count=('mean_efficiency', 'count')
).round(3)
print(cluster_summary)

Section-category pairs for clustering: 891

=== Elbow method ===
  k      inertia   silhouette
  2       2084.1       0.4160
  3       1330.7       0.4320
  4        966.6       0.4601
  5        814.7       0.4455
  6        698.4       0.4163
  7        633.8       0.3832
  8        585.7       0.3746

=== Cluster profiles ===
         mean_efficiency  std_efficiency  pct_months_below_70  count
cluster                                                             
0                  0.524           0.477                0.719    275
1                  0.610           0.534                0.624    148
2                  0.060           0.083                0.991    214
3                  0.979           0.296                0.125    254


In [11]:
import joblib

# Label clusters
cluster_labels = {2: 'High Risk', 0: 'Medium Risk', 1: 'Moderate Risk', 3: 'Low Risk'}
profile['risk_tier'] = profile['cluster'].map(cluster_labels)

# Verify labeling
print("=== Risk tier distribution ===")
print(profile['risk_tier'].value_counts())

print("\n=== Risk tier vs category group ===")
print(pd.crosstab(profile['risk_tier'], profile['category_group']))

# Save section risk tiers CSV
PROCESSED = ROOT / "data" / "processed"
profile[['section', 'category_group', 'risk_tier', 
          'mean_efficiency', 'std_efficiency', 
          'pct_months_below_70']].to_csv(
    PROCESSED / "section_risk_tiers.csv", index=False
)
print("\nSaved: data/processed/section_risk_tiers.csv")

# Save the fitted KMeans and scaler — critical for inference later
MODELS = ROOT / "models"
joblib.dump(kmeans, MODELS / "kmeans_risk_tier.pkl")
joblib.dump(scaler, MODELS / "kmeans_scaler.pkl")
joblib.dump(features, MODELS / "kmeans_features.pkl")
print("Saved: models/kmeans_risk_tier.pkl")
print("Saved: models/kmeans_scaler.pkl")
print("Saved: models/kmeans_features.pkl")

# Quick sanity check — where do our chronic underperformers land?
print("\n=== Risk tier for known problem sections ===")
check_sections = ['MALLARAM WTP (Div 21 - GHANPUR)', 'KPHB', 
                  'VAISHALINAGAR (DIV 10)', 'BANJARA HILLS']
print(profile[profile['section'].isin(check_sections)][
    ['section', 'category_group', 'risk_tier', 'mean_efficiency']
].to_string(index=False))

=== Risk tier distribution ===
risk_tier
Medium Risk      275
Low Risk         254
High Risk        214
Moderate Risk    148
Name: count, dtype: int64

=== Risk tier vs category group ===
category_group  Commercial  Domestic  GovtInstitutional  Industrial  Slum
risk_tier                                                                
High Risk                2         7                 30           5   170
Low Risk               135        16                 21          80     2
Medium Risk             28       183                 51          11     2
Moderate Risk           33         4                 81          20    10

Saved: data/processed/section_risk_tiers.csv
Saved: models/kmeans_risk_tier.pkl
Saved: models/kmeans_scaler.pkl
Saved: models/kmeans_features.pkl

=== Risk tier for known problem sections ===
                        section    category_group     risk_tier  mean_efficiency
                  BANJARA HILLS        Commercial      Low Risk         1.007779
             

## EDA complete — summary for ETL and feature engineering

### Data facts
- 127,482 total rows across 52 monthly billing files (2022–2026)
- 8 columns, already long format — no wide-format reshaping needed
- 21.4% zero-demand rows — excluded from efficiency computation
- 221 null division rows (CHANDANAGAR, SAHEBNAGAR) — fill with 0
- 29 completely empty rows — drop

### Target variable
- collection_efficiency = collection / demand, clipped to [0, 2]
- Mean: 0.557, Median: 0.487
- 51% of valid rows have efficiency below 0.5
- Total shortfall: ₹31.97 billion

### Key findings
1. Domestic drives shortfall: 44.8% of demand, 65.8% of shortfall (₹21B)
2. January anomaly: ₹9.5B shortfall in January alone (30% of total)
3. Seasonal assumption wrong: monsoon months slightly BETTER than average
4. Industrial pays reliably (100% median), Slum essentially never pays (1.4%)
5. 344 chronic underperformer section-category pairs identified

### Clustering (k=4, silhouette=0.460)
- High Risk (214 pairs): dominated by Slum + structural zeros
- Medium Risk (275 pairs): mostly Domestic — highest business value
- Moderate Risk (148 pairs): GovtInstitutional with variable payment
- Low Risk (254 pairs): Commercial + Industrial reliable payers
- Artifacts saved: models/kmeans_risk_tier.pkl, kmeans_scaler.pkl

### Feature engineering implications
- lag_1_efficiency will be strongest predictor
- category_group will be second strongest
- is_monsoon and is_financial_year_end will have weak signal
- January indicator (is_january) should be added — stronger than monsoon flag
- risk_tier join will add meaningful signal for domestic sections

In [13]:
jupyter nbconvert --to script notebooks/01_eda.ipynb --output notebooks/01_eda_checkpoint

SyntaxError: invalid decimal literal (4063757329.py, line 1)